# Differentiable cut optimisation for a Z' search

**Hackathon notebook.** You will tune the selection of a Z' $\to t\bar t$ search by gradient descent, instead of scanning cut values by hand.

The big idea fits in one sentence. A hard cut has zero gradient, a soft cut has a useful one, and a useful gradient lets a computer improve the cut for you.

**How this notebook works**
- Cells marked **TODO** need you to fill in a few lines. Everything else is provided.
- The first plot is written out in full. Later plotting cells only describe what to draw; writing the plot is part of the exercise.
- We start from a flat table of events, read from the skimmed CMS Open Data samples in `data/`.
- An optional part shows how to read real CMS Open Data NanoAOD files with `uproot`. The rest of the notebook does not depend on it.

**Plan**
1. Look at the events and set up the data.
2. Baseline: optimise a hard cut by scanning.
3. See why gradients fail for a hard cut.
4. Replace the hard cut by a soft cut and optimise it by gradient ascent.
5. Build a soft histogram and optimise the bin edges.
6. Optimise several cuts together and compare with a brute-force scan.
7. Check the result honestly on events the optimiser never saw.
8. Stretch goals.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

# uproot and awkward are only needed by the optional load_nano_events
import uproot
import awkward as ak

# JAX uses 32-bit floats by default. 64-bit avoids rounding surprises when we sum many small weights.
jax.config.update("jax_enable_x64", True)

plt.rcParams.update({"figure.figsize": (6.5, 3.8), "axes.grid": True, "grid.alpha": 0.3})

## 1. The events

Each event has three numbers.

- `m_ttbar`: the reconstructed mass of the top-quark pair, in GeV. A heavy resonance decaying to top quarks shows up as a bump on a falling background.
- `leading_jet_pt`: the transverse momentum of the highest-pT jet, in GeV.
- `met_pt`: the missing transverse momentum, in GeV.

Each event also has a **weight**. A weight says how many events of that kind we expect in real data. Summing the weights of the events that pass a selection gives the expected yield, called $s$ for signal and $b$ for background.

The significance we want to maximise is

$$Z = \frac{s}{\sqrt{b}} .$$

A larger $Z$ means the signal would stand out more against the background fluctuations.

### 1a. The samples

The folder `data/` holds one `.npz` file per sample, skimmed from CMS Open Data NanoAOD. `signal` is a Z' boson with a mass of about 2 TeV decaying to $t\bar t$. The backgrounds are two $t\bar t$ samples (`ttbar`, `ttbar_lep`) and W boson production with jets (`wjets`). Each file is a flat table with one entry per event, and the event weights add up to the expected yield of that sample. The file `data.npz` holds real collision events with unit weights; we do not use it in this exercise.

`load_sample` reads one file. The optional `load_nano_events` below shows how a NanoAOD file is read directly with `uproot`; nothing later in the notebook depends on it.

In [ ]:
SAMPLES = {"signal": True, "ttbar": False, "ttbar_lep": False, "wjets": False}


def load_sample(name, is_signal):
    """Read one npz file: flat arrays, one entry per event."""
    d = np.load(f"data/{name}.npz")
    out = {k: d[k] for k in ("m_ttbar", "leading_jet_pt", "met_pt")}
    out["w"] = d["weight"]
    out["is_sig"] = np.full(len(d["weight"]), is_signal)
    return out


parts = [load_sample(name, is_sig) for name, is_sig in SAMPLES.items()]
events = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}

for name, p in zip(SAMPLES, parts):
    print(f"{name:10s}: {len(p['w']):7d} events, expected yield {float(p['w'].sum()):7.1f}")

In [ ]:
def load_nano_events(path, xsec_pb, lumi_fb, is_signal, max_events=None):
    """Read a NanoAOD file and return flat arrays, one entry per selected event.

    Preselection: at least one muon with pT > 30 GeV and at least two jets with pT > 30 GeV.
    Weights are scaled so that they add up to  cross-section x luminosity  before the selection.
    """
    with uproot.open(path) as f:
        a = f["Events"].arrays(["Jet_pt", "Muon_pt", "MET_pt", "genWeight"], entry_stop=max_events)

    jet_pt = a["Jet_pt"]
    good_jets = jet_pt[jet_pt > 30.0]

    ht = ...                                                          # TODO: scalar sum of the good jet pT
    lep_pt = ...                                                      # TODO: pT of the leading muon (0 if none)

    keep = (lep_pt > 30.0) & (ak.num(good_jets, axis=1) >= 2)

    w = a["genWeight"]
    w = w * (xsec_pb * 1000.0 * lumi_fb / ak.sum(w))                  # cross-section in pb -> fb, times luminosity in 1/fb

    n_keep = int(ak.sum(keep))
    return {
        "ht": ak.to_numpy(ht[keep]),
        "lep_pt": ak.to_numpy(lep_pt[keep]),
        "met": ak.to_numpy(a["MET_pt"][keep]),
        "w": ak.to_numpy(w[keep]),
        "is_sig": np.full(n_keep, is_signal),
    }

### 1b. Scale the variables and split the events in two

We rescale the variables so that typical values are of order 1. That makes cut values and learning rates easier to reason about. `m_ttbar` and the jet pT are in TeV, the missing momentum in units of 100 GeV.

We also split the events into a **training** half and a **test** half. The optimiser only sees the training half. The test half is kept for the honest check in section 7.

Signal and background weights are stored in separate arrays. A weight is zero in the array of the other class. That way `w_s * selection` summed over events is the signal yield, and the same for the background.

In [ ]:
SCALE = {"m_ttbar": 1000.0, "leading_jet_pt": 1000.0, "met_pt": 100.0}
FEATURES = ["m_ttbar", "leading_jet_pt", "met_pt"]
X_all = np.stack([events[f] / SCALE[f] for f in FEATURES], axis=1)

order = np.random.default_rng(0).permutation(len(X_all))
half = len(order) // 2


def make_dataset(idx):
    factor = len(order) / len(idx)           # each half stands in for the full expected yield
    is_sig, w = events["is_sig"][idx], events["w"][idx] * factor
    return {
        "X": jnp.array(X_all[idx]),
        "w_s": jnp.array(np.where(is_sig, w, 0.0)),
        "w_b": jnp.array(np.where(is_sig, 0.0, w)),
    }


train = make_dataset(order[:half])
test = make_dataset(order[half:])
print(f"train: {train['X'].shape[0]} events, expected signal {float(train['w_s'].sum()):.1f}, background {float(train['w_b'].sum()):.0f}")
print(f"test:  {test['X'].shape[0]} events, expected signal {float(test['w_s'].sum()):.1f}, background {float(test['w_b'].sum()):.0f}")

In [ ]:
bins = np.linspace(0.2, 4.0, 50)
x_m = np.asarray(train["X"][:, 0])

fig, ax = plt.subplots()
ax.hist(x_m, bins=bins, weights=np.asarray(train["w_b"]), histtype="step", lw=2, label="background")
ax.hist(x_m, bins=bins, weights=np.asarray(train["w_s"]), histtype="step", lw=2, label="signal")
ax.set_yscale("log")
ax.set_xlabel("m_ttbar [TeV]")
ax.set_ylabel("expected events per bin")
ax.legend()
plt.show()

**Question.** Looking at the plot, roughly where would you put a cut on `m_ttbar`? Write down your guess. We will compare it with the optimiser's answer later.

## 2. Baseline: a hard cut and a scan

A hard cut keeps an event if `m_ttbar` is above some value and throws it away otherwise. We describe a selection by a number between 0 and 1 for each event: its **selection weight**. For a hard cut that number is exactly 0 or 1.

The helper `yields` is provided. It sums the weights of the events that pass, split into signal and background.

In [ ]:
B_FLOOR = 1.0     # explained in the note below


def yields(sel, d):
    """Expected signal and background after a selection.

    sel is one number per event: 1 keeps the event, 0 drops it, values in between keep part of it.
    """
    return jnp.sum(d["w_s"] * sel), jnp.sum(d["w_b"] * sel)

**A note on `B_FLOOR`.** The simulation has a finite number of background events. If a selection leaves almost none, $b$ can end up close to zero by accident, and $s/\sqrt{b}$ becomes enormous. That is a statistical accident of the simulation, not real sensitivity, and an optimiser will happily hunt for it. Adding one expected background event inside the square root removes the problem. It makes no visible difference when $b$ is large.

**TODO.** Fill in the two functions below.

- `significance(s, b)` returns $s/\sqrt{b + b_{\text{floor}}}$, with `B_FLOOR` defined above. The floor is explained right after this cell.
- `Z_hard(cut, d)` applies the hard cut `m_ttbar > cut` to dataset `d` and returns the significance. The `m_ttbar` values are in `d["X"][:, 0]`.

In [ ]:
def significance(s, b):
    ...


def Z_hard(cut, d):
    ...

In [ ]:
cuts = np.linspace(0.3, 3.0, 111)
Z_scan = np.array([float(Z_hard(c, train)) for c in cuts])
best_cut = cuts[Z_scan.argmax()]

print(f"best hard cut from the scan: m_ttbar > {best_cut:.2f} TeV, Z = {Z_scan.max():.2f}")

In [ ]:
# Your plot: Z_scan against cuts, with a vertical line at best_cut.
# The histogram in section 1b shows the matplotlib pattern to follow.
...

This works for one cut. With three cuts and a fine grid it already needs hundreds of thousands of evaluations. With a dozen cuts it is hopeless. We want to follow the slope of $Z$ uphill instead of visiting every point.

## 3. Why a hard cut cannot be optimised by gradient

JAX can compute the derivative of $Z$ with respect to the cut value with `jax.grad`. Ask it for the derivative at a cut of 1.5 TeV, where the curve above clearly has a slope.

In [ ]:
grad_hard = jax.grad(Z_hard)(1.5, train)
print("dZ/dcut from jax.grad:", float(grad_hard))

**Question.** The plot above shows a clear slope at 1.5 TeV, yet the gradient is exactly zero. Why? Think about what happens to a single event when the cut moves by a tiny amount.

## 4. A soft cut

Replace the step function by a smooth one, the **sigmoid**:

$$ \text{sel}(x) = \sigma\!\left(\frac{x - c}{\tau}\right), \qquad \sigma(u) = \frac{1}{1 + e^{-u}} . $$

Far above the cut the selection weight is close to 1, far below it is close to 0, and near the cut it changes smoothly. The number $\tau$ is the **width** of the transition. A small $\tau$ looks like a hard cut. A large $\tau$ is very blurry.

**TODO.** Write `soft_cut` and `Z_soft`. Use `jax.nn.sigmoid`.

In [ ]:
def soft_cut(x, cut, tau):
    ...


def Z_soft(cut, d, tau):
    ...

In [ ]:
# Your plot: the hard cut at 1.0 TeV and soft_cut(xs, 1.0, tau) for tau in [0.02, 0.1, 0.3],
# drawn as selection weight against xs = np.linspace(0.0, 2.0, 400).
...

Now the significance as a function of the cut, for the hard cut and for soft cuts of different widths.

In [ ]:
# Your plot: the hard-cut Z_scan together with the soft Z_soft(c, train, tau)
# for tau in [0.02, 0.1, 0.3], each evaluated over the same cuts array.
...

The soft curves are smooth and have a slope everywhere. Check that `jax.grad` now gives a non-zero answer.

In [ ]:
print("dZ/dcut at 1.0 TeV:", float(jax.grad(Z_soft)(1.0, train, 0.1)))
print("dZ/dcut at 1.5 TeV:", float(jax.grad(Z_soft)(1.5, train, 0.1)))
print("dZ/dcut at 2.4 TeV:", float(jax.grad(Z_soft)(2.4, train, 0.1)))

### Gradient ascent

To maximise $Z$ we repeat one step many times:

$$ p \leftarrow p + \eta \, \frac{\partial Z}{\partial p} . $$

Here $p$ holds the parameters being tuned and $\eta$ is the **learning rate**. Each step moves the parameters a little way uphill.

**TODO.** Write the loop in `gradient_ascent`. The gradient function `grad_f` is already built for you. Store every value of `p` in `path` so we can plot the journey.

In [ ]:
def gradient_ascent(f, p0, lr, n_steps):
    grad_f = jax.jit(jax.grad(f))
    p = p0
    path = [p]
    # TODO: repeat n_steps times: move p one step uphill, then append it to path
    ...
    return p, np.array(path)

In [ ]:
TAU = 0.1

p_opt, path = gradient_ascent(lambda c: Z_soft(c, train, TAU), jnp.array(0.8), lr=0.02, n_steps=150)

print(f"optimised cut:  m_ttbar > {float(p_opt):.2f} TeV")
print(f"soft Z at that cut: {float(Z_soft(p_opt, train, TAU)):.2f}")
print(f"HARD Z at that cut: {float(Z_hard(p_opt, train)):.2f}   (scan optimum: {Z_scan.max():.2f} at {best_cut:.2f} TeV)")

In [ ]:
# Your plot: path against the step number, with a horizontal line at best_cut.
...

**Things to try**
- Change the learning rate. What happens with `lr=0.002`? With `lr=0.2`?
- Change the starting point `0.8` to `0.4` and to `2.4`. Do you reach the same answer?
- Change `TAU` to `0.02`. The optimised cut lands even closer to the scan optimum. Why can we afford such a small width with this many events?
- Change `TAU` to `0.5`. The optimum moves. Which cut value do you get now, and what is its **hard** $Z$?

The width $\tau$ is a trade-off. If it is too small the gradient vanishes away from the cut. If it is too large the soft cut no longer resembles the hard cut you will actually use. The loop below shows the effect for a range of widths.

In [ ]:
print(" width |  optimised cut | hard Z at that cut")
for tau in [0.02, 0.05, 0.1, 0.2, 0.4]:
    p, _ = gradient_ascent(lambda c: Z_soft(c, train, tau), jnp.array(1.0), lr=0.02, n_steps=300)
    print(f" {tau:5.2f} | {float(p):14.2f} | {float(Z_hard(p, train)):.2f}")

## 5. A soft histogram and its bin edges

Real analyses fit a histogram, not a single cut. A histogram has bin edges, and bin edges are cuts too. We can make a histogram differentiable the same way.

An event belongs to bin $i$ if it is above edge $e_i$ and below edge $e_{i+1}$. As soft cuts that is

$$ \text{membership}_i(x) = \sigma\!\left(\frac{x - e_i}{\tau}\right) - \sigma\!\left(\frac{x - e_{i+1}}{\tau}\right), $$

with $\sigma = 1$ for the lowest edge (at $-\infty$) and $\sigma = 0$ for the highest edge (at $+\infty$). Summing the signal weights times the membership gives the signal yield of each bin, and the same for the background.

Each bin has its own significance $s_i/\sqrt{b_i}$. Independent bins combine in quadrature:

$$ Z = \sqrt{\sum_i \frac{s_i^2}{b_i}} . $$

**TODO.** Complete `soft_bin_membership` and `Z_binned`. The `edges` array holds the inner edges. Sorting it keeps the bins in order even if the optimiser moves two edges past each other.

In [ ]:
def soft_bin_membership(x, edges, tau):
    """Array of shape (n_events, n_bins) saying how much each event belongs to each bin."""
    ...


def Z_binned(edges, d, tau):
    ...

In [ ]:
TAU_BIN = 0.1
edges0 = jnp.array([0.8, 1.4, 2.0])          # starting edges: four bins

edges_opt, edge_path = gradient_ascent(lambda e: Z_binned(e, train, TAU_BIN), edges0, lr=0.02, n_steps=300)
edges_opt = jnp.sort(edges_opt)

print("starting edges :", np.round(np.asarray(edges0), 2), f"  Z = {float(Z_binned(edges0, train, TAU_BIN)):.2f}")
print("optimised edges:", np.round(np.asarray(edges_opt), 2), f"  Z = {float(Z_binned(edges_opt, train, TAU_BIN)):.2f}")
print(f"best single hard cut for comparison: Z = {Z_scan.max():.2f}")

In [ ]:
# Your plot: edge_path against the step number (one curve per edge).
...

**Question.** The binned $Z$ is at least as large as the single-cut $Z$ of the scan. Why can adding bins never make the sensitivity worse?

**Things to try**
- Use five or six bins by giving more starting edges.
- Where do the optimised edges end up? Do they pile up near the signal region?
- Evaluate the optimised edges with a very small `TAU_BIN`, such as `0.01`, to see what the histogram looks like with sharp bins.

## 6. Several cuts at once

So far we tuned one number. The real advantage of gradients appears when many numbers are tuned together.

Take a cut on `m_ttbar` and a cut on the leading jet pT. A selection that passes both cuts has weight $\sigma_1 \times \sigma_2$, the product of two soft cuts. Each cut has its own width.

**TODO.** Write `soft_selection`. The array `cuts` has one entry per cut and uses the first `len(cuts)` columns of `X`. The result must have one number per event.

In [ ]:
def soft_selection(X, cuts, taus):
    ...


def hard_selection(X, cuts):
    return jnp.all(X[:, :cuts.shape[0]] > cuts[None, :], axis=1).astype(float)


def Z_joint_soft(cuts, d, taus):
    s, b = yields(soft_selection(d["X"], cuts, taus), d)
    return significance(s, b)


def Z_joint_hard(cuts, d):
    s, b = yields(hard_selection(d["X"], cuts), d)
    return significance(s, b)

For comparison we need the brute-force answer. This helper scans a grid of hard cuts on m_ttbar and the leading jet pT. It is provided, so you can just run it.

In [ ]:
def scan_2d(d, grid1, grid2):
    one_row = jax.jit(jax.vmap(lambda c2, c1: Z_joint_hard(jnp.array([c1, c2]), d), in_axes=(0, None)))
    return np.array([one_row(jnp.asarray(grid2), c1) for c1 in grid1])       # shape (len(grid1), len(grid2))


grid_m = np.linspace(0.3, 2.6, 45)
grid_jet = np.linspace(0.15, 1.2, 50)
Z_grid = scan_2d(train, grid_m, grid_jet)

i, j = np.unravel_index(Z_grid.argmax(), Z_grid.shape)
print(f"brute-force scan ({Z_grid.size} points): m_ttbar > {grid_m[i]:.2f} TeV, leading jet pT > {1000*grid_jet[j]:.0f} GeV, Z = {Z_grid.max():.2f}")

In [ ]:
TAUS = jnp.array([0.1, 0.05])
cuts0 = jnp.array([0.8, 0.2])

cuts_opt, cut_path = gradient_ascent(lambda c: Z_joint_soft(c, train, TAUS), cuts0, lr=0.01, n_steps=500)

print(f"gradient ascent: m_ttbar > {float(cuts_opt[0]):.2f} TeV, leading jet pT > {1000*float(cuts_opt[1]):.0f} GeV")
print(f"soft Z = {float(Z_joint_soft(cuts_opt, train, TAUS)):.2f},  hard Z = {float(Z_joint_hard(cuts_opt, train)):.2f}")

In [ ]:
# Your plot: Z_grid as a heat map, ax.pcolormesh(grid_m, grid_jet, Z_grid.T, shading="auto"),
# with the gradient-ascent path cut_path drawn on top (start, trajectory, end).
...

**Questions**
- How many times did the brute-force scan evaluate $Z$, and how many gradient steps did the optimiser need?
- Now imagine ten cuts instead of two. How would the cost of each method change?
- Does the jet pT cut help at all compared with the `m_ttbar` cut alone? Compare the two best $Z$ values.

**Things to try**
- Add the MET cut as a third cut by giving three starting values and three widths. The brute-force scan cannot easily be drawn any more, but the optimiser does not care.
- Try different starting points. Does the optimiser always end up in the same place? Start the jet pT cut at `2.5` and watch it get stuck. Almost no events sit near a cut that high, so the gradient is tiny. This is the same vanishing-gradient problem as a very small width.

## 7. An honest evaluation

Everything so far used the training events, both for choosing the cuts and for reporting $Z$. That is a classic trap. Cuts tuned on a sample can exploit its random fluctuations, and then look better on that sample than they will on new data.

The fix is simple. Tune on the training half, then evaluate the final **hard** cuts on the test half, which the optimiser has never seen.

With 100 000 training events and two cuts, the effect is tiny. To see it clearly, we make the training sample small. The helper `subsample` keeps `n` random training events and rescales their weights so that the total expected yield stays the same.

In [ ]:
def subsample(d, n, seed=0):
    N = d["X"].shape[0]
    idx = np.random.default_rng(seed).choice(N, n, replace=False)
    f = N / n
    return {"X": d["X"][idx], "w_s": d["w_s"][idx] * f, "w_b": d["w_b"][idx] * f}

**TODO.** For each training size, tune the two cuts by gradient ascent on the small training sample, then report the **hard-cut** $Z$ on that same sample and on the full test half.

In [ ]:
def tune_and_evaluate(d_train, d_test, seed_cuts=jnp.array([0.8, 0.2])):
    # TODO: tune the two cuts by gradient ascent on d_train (lr=0.01, 500 steps),
    # then return the cuts and the hard-cut Z on d_train and on d_test
    ...


for n in [300, 1000, 10_000, 70_000]:
    z_tr, z_te = [], []
    for seed in range(10):
        small = subsample(train, n, seed=seed)
        _, a, b = tune_and_evaluate(small, test)
        z_tr.append(a)
        z_te.append(b)
    print(f"n_train = {n:7d}:  Z on training sample = {np.mean(z_tr):.2f},  Z on test sample = {np.mean(z_te):.2f}")

**Questions**
- For which training sizes is the training $Z$ larger than the test $Z$? What does that gap tell you?
- With only a few parameters the gap stays small for large samples. What would you expect if you tuned hundreds of parameters on the same sample?
- The brute-force scan overfits in exactly the same way. Is gradient-based tuning more or less vulnerable to it?

## Wrap-up

You have gone from a cut scan to a small differentiable analysis.

- A hard cut has no useful gradient. A soft cut does.
- The width of the soft cut trades a usable gradient against faithfulness to the hard cut.
- Binning can be made differentiable the same way, so bin edges become parameters.
- Many parameters can be tuned together by following one gradient.
- Always evaluate the final **hard** selection on events that were not used for tuning.

This is the same idea that full-scale differentiable analyses apply to a complete selection, a neural-network classifier and a statistical model together.

## 8. Stretch goals

Pick whichever interests you. Each uses only functions you have already written.

### A. A better significance, with a background uncertainty

The formula $s/\sqrt{b}$ is a simplification. Two common improvements:

$$ Z_{\text{Asimov}} = \sqrt{2\left[(s+b)\ln\!\left(1 + \frac{s}{b}\right) - s\right]}, \qquad
Z_{\delta} = \frac{s}{\sqrt{b + (\delta\, b)^2}} . $$

The second one includes a fractional uncertainty $\delta$ on the background yield, for example $\delta = 0.1$ for 10 percent.

**Task.** Redefine `significance` with each formula in turn and rerun the single-cut optimisation of section 4. Does the optimal cut move? In which direction, and why?

In [ ]:
def significance_asimov(s, b):
    ...


def make_significance_delta(delta):
    ...


# TODO: rerun the single-cut optimisation of section 4 with each formula
# and compare the optimised cut positions

### B. A learned selection instead of rectangular cuts

Cuts on m_ttbar, the jet pT and MET make a rectangular box in feature space. A **linear discriminant** can tilt the boundary:

$$ \text{sel}(\vec{x}) = \sigma\!\left(\vec{w}\cdot\vec{x} + w_0\right) . $$

The three numbers in $\vec w$ and the offset $w_0$ are the parameters. Tune them directly against $Z$ with the same gradient ascent. This is a one-neuron neural network trained on the physics goal instead of on a generic loss.

**Task.** Write `Z_linear(params, d)` with `params = [w1, w2, w3, w0]`. Tune it, then evaluate the corresponding hard selection (`w . x + w0 > 0`) on the test half and compare with the rectangular cuts of section 6. This needs many more steps than a single cut, so be patient and plot $Z$ against the step number if you want to see it converge.

In [ ]:
def Z_linear(params, d):
    ...


def Z_linear_hard(params, d):
    ...


# TODO: tune the four parameters with gradient_ascent (a few thousand steps),
# then compare the hard Z on train and test with the rectangular cuts of section 6

### C. Read NanoAOD directly

The npz files in `data/` were skimmed from CMS Open Data NanoAOD. The optional function `load_nano_events` in section 1a shows how such a file is read with `uproot`: it returns flat arrays for HT, the leading muon pT and the MET, with weights scaled to cross-section times luminosity. Fill in its two missing lines if you have not already.

**Task.** Point it at NanoAOD files for a signal and a background, rebuild `events` from its output, and set `FEATURES` and `SCALE` to the variables it returns. Then rerun the whole notebook.

- Do the optimal cut values move? Why?
- Is the soft width $\tau$ still sensible for the new variables?
- Which of the three variables matters most?